# MangGO — Train & Export Mango Defect DetectorRuns entirely on your Mac. Apple Silicon trains on the GPU through PyTorch MPS,and Core ML export works locally because this is macOS.**Before running:** set up the environment and pick it as the kernel — see`README.md` in this folder. Top right of VS Code → Select Kernel → `.venv`.

## 0. Environment check`coremltools` does not support the newest Python releases right away. If theversion printed below is 3.13 or newer and the export cell fails later, rebuildthe venv with `python3.12`.

In [ ]:
import platform, sysprint("python  ", sys.version.split()[0])print("machine ", platform.machine())print("macOS   ", platform.mac_ver()[0])

In [ ]:
%pip install -q ultralytics roboflow coremltools

In [ ]:
import os# Lets unsupported operations fall back to CPU instead of crashing the run.os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"import torchDEVICE = "mps" if torch.backends.mps.is_available() else "cpu"print("torch  ", torch.__version__)print("device ", DEVICE)if DEVICE == "cpu":    print("\nGPU unavailable — training will be slow. Lower EPOCHS to about 40.")

## 1. Download the datasetTwo things to fill in:- `API_KEY` — from <https://app.roboflow.com/settings/api>- `VERSION` — open the dataset page, click **Download Dataset** → **Show  download code**, and copy the number inside `version(...)`The dataset lands in a folder next to this notebook.

In [ ]:
from roboflow import RoboflowAPI_KEY = "PASTE_YOUR_ROBOFLOW_API_KEY"WORKSPACE = "mangotest"PROJECT = "mango-defect-detectionv4"VERSION = 15dataset = (    Roboflow(api_key=API_KEY)    .workspace(WORKSPACE)    .project(PROJECT)    .version(VERSION)    .download("yolov11"))DATA_YAML = f"{dataset.location}/data.yaml"print(DATA_YAML)

In [ ]:
print(open(DATA_YAML).read())

## 2. Train`yolo11n` is the smallest variant, which is what you want for on-deviceinference.`patience` stops the run early once validation stops improving, so the datasetbeing small does not turn into 100 epochs of overfitting.`workers=0` keeps data loading in this process. Background workers can hanginside Jupyter on macOS, and with a dataset this small they buy little.Raise `batch` to 16 if you have plenty of memory, or drop it to 4 if the runruns out.

In [ ]:
from ultralytics import YOLOEPOCHS = 100BATCH = 8model = YOLO("yolo11n.pt")model.train(    data=DATA_YAML,    epochs=EPOCHS,    patience=25,    imgsz=640,    batch=BATCH,    device=DEVICE,    workers=0,    cache=True,    project="runs",    name="mango-defect",    exist_ok=True,    seed=0,)

## 3. MetricsPublished baseline for this dataset: mAP@50 **38.2%**, precision **43.6%**,recall **42.0%**.Landing near that is expected. The ceiling here is the dataset — 401 images witha single class — not the training setup.

In [ ]:
metrics = model.val(data=DATA_YAML, device=DEVICE, workers=0)print(f"mAP@50     {metrics.box.map50:.3f}")print(f"mAP@50-95  {metrics.box.map:.3f}")print(f"precision  {metrics.box.mp:.3f}")print(f"recall     {metrics.box.mr:.3f}")

In [ ]:
from IPython.display import Image, displayfor name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:    path = f"runs/mango-defect/{name}"    if os.path.exists(path):        print(name)        display(Image(filename=path, width=760))

## 4. Export to Core ML`nms=True` embeds the non-max-suppression pipeline. Without it Vision hands backraw MultiArrays instead of `VNRecognizedObjectObservation`, and the Swift sidereceives nothing.

In [ ]:
from pathlib import PathWEIGHTS = Path("runs/mango-defect/weights/best.pt")assert WEIGHTS.exists(), f"not found: {WEIGHTS.resolve()}"exported = Path(YOLO(str(WEIGHTS)).export(format="coreml", nms=True, imgsz=640))print(exported)

## 5. Install into the appWalks up from this notebook to find the repository root, then drops the modelwhere `CoreMLDefectDetector` looks for it.

In [ ]:
import shutildef find_repo_root(start: Path) -> Path:    for folder in [start, *start.parents]:        if (folder / "MangGO" / "MangGO" / "Core").is_dir():            return folder    raise FileNotFoundError("repository root not found — run this notebook from inside the repo")destination = find_repo_root(Path.cwd()) / "MangGO/MangGO/Core/Vision/Resources/MangoDefect.mlpackage"destination.parent.mkdir(parents=True, exist_ok=True)if destination.exists():    shutil.rmtree(destination)shutil.move(str(exported), str(destination))print("installed →", destination)

## DoneBack in Xcode:1. Confirm `MangoDefect.mlpackage` shows up under `Core/Vision/Resources`, and   that it appears in **Build Phases → Compile Sources**2. Open it and check the Metadata tab says **Object Detector** with `Confidence`   and `Coordinates` outputs3. In `iPhoneView.swift`, swap `CaptureView()` for   `CaptureView(model: CaptureViewModel(detector: CoreMLDefectDetector()))`4. Build to a physical iPhone — the Simulator has no camera